# cross_validation_reg

> 对应代码：`EconometricsCode/code_in_notes/Chap.2/cross_validation_reg.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.2`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.2")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们先清空内存并设定 50 个观测、固定随机种子，保证模拟结果可以复现。

In [2]:
%%stata
clear
set obs 50
set seed 1000


. clear

. set obs 50
Number of observations (_N) was 0, now 50.

. set seed 1000

. 


生成解释变量 x（取自 [0,3] 上的均匀分布）和被解释变量 y：真实的条件期望函数是指数函数 exp(x)，再叠加一个标准差为 2 的正态扰动。这是一个非线性的数据生成过程，我们故意用多项式去逼近它，以便观察模型复杂度对样本外预测的影响。

In [3]:
%%stata
gen x=runiform()*3
gen y=exp(x)+rnormal()*2


. gen x=runiform()*3

. gen y=exp(x)+rnormal()*2

. 


初始化一个空的局部宏 control 用来逐步累积多项式项，scalar K 记录当前模型的阶数（此处未实际使用，仅为示意）。

In [4]:
%%stata
local control ""
scalar K=0


. local control ""

. scalar K=0

. 


外层循环让多项式阶数 i 从 1 增加到 10：每次生成 x 的 i 次幂，并把它追加到解释变量列表 control 中，这样第 i 轮回归使用的就是 1 到 i 阶的多项式。

这是留一法交叉验证（leave-one-out cross-validation, LOOCV）的核心：对样本中第 j 个观测，我们把它剔除后用其余 49 个观测回归（if _n~=`j'），再用 predict 得到全部样本的残差，只保留被剔除的那个观测的残差存入 error。这样每个观测的预测误差都来自一个没见过它的模型，因而可以近似样本外预测误差。整个过程放在 quietly 块里抑制中间输出。

把留一法得到的预测误差平方后取平均，就得到该阶多项式的交叉验证均方误差（CV MSE），它是对样本外预测风险的估计。我们把阶数 i 和对应的 MSE 打印出来比较：MSE 最小的阶数就是交叉验证选出的最优模型。最后清掉中间变量，进入下一阶的循环。

In [5]:
%%stata
forvalues i=1/10{
  gen x`i'=x^`i'
  local control "`control' x`i'"
  local N= _N
  quietly{
    gen error=.
    forvalues j=1/`N'{
      reg y `control' if _n~=`j'
      predict resid, residual
      replace error=resid if _n==`j'
      drop resid
    }
  }
  gen error2=error^2
  quietly: su error2
  scalar mse=r(mean)
  display `i' _skip mse
  drop error error2
}


. forvalues i=1/10{
  2.   gen x`i'=x^`i'
  3.   local control "`control' x`i'"
  4.   local N= _N
  5.   quietly{
  6.     gen error=.
  7.     forvalues j=1/`N'{
  8.       reg y `control' if _n~=`j'
  9.       predict resid, residual
 10.       replace error=resid if _n==`j'
 11.       drop resid
 12.     }
 13.   }
 14.   gen error2=error^2
 15.   quietly: su error2
 16.   scalar mse=r(mean)
 17.   display `i' _skip mse
 18.   drop error error2
 19. }
1 7.4712101
2 3.2677203
3 3.1873242
4 3.2950048
5 3.4450243
6 3.6555565
7 3.6281509
8 3.5861068
9 3.8194943
10 3.8540827

. 
